# Lab 29: Measuring AI Security: Outcomes, Not Motion

*AI Cybersecurity Masterclass · Lab 29 · AI60secLab*

**Companion video:** [▶ How to Measure AI Security in 60 Seconds](https://www.youtube.com/watch?v=DcbG2ol3XCQ&list=PLG4l6Ssi9A00)

**Research question:** Is the AI security program getting safer, or only producing more dashboards?

We build a layered scorecard from measurements the earlier labs taught you to take. Every metric has a definition, a baseline, a threshold, an owner, and a decision attached to it.

**Learning objectives**
- tell motion metrics from outcome metrics;
- measure across layers: detection, robustness, LLM application, agent, privacy, operations;
- balance missed risk against alert fatigue;
- attach a baseline, threshold, owner, and decision to every metric;
- read a scorecard as a list of next actions.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics
from src.adversarial import available_numeric_attack_features, training_bounds, bounded_random_perturbation
from src.llm_app import ControlConfig, build_test_cases, evaluate_controls, CANARY_SECRET
from src.privacy_attacks import simulate_assistant_logging, scan_for_canaries
from src.redteam import APP_VERSIONS, default_cases, run_campaign
from src.ai_metrics import Metric, scorecard, motion_metrics_example

## Step 1: Motion versus outcome

In [ ]:
display(motion_metrics_example())

## Step 2: Take the measurements

Each block is a small, fast version of an earlier lab.

In [ ]:
# Detection quality (Labs 02, 06, 09)
train, test = load_unsw(DATA_DIR)
train_sample = train.sample(40000, random_state=0)
X_train, y_train = split_xy(train_sample, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)
X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

detector = random_forest_pipeline(X_train)
detector.set_params(model__n_estimators=100)
detector.fit(X_train, y_train)
scores = detector.predict_proba(X_test)[:, 1]
detection = binary_metrics(y_test, (scores >= 0.5).astype(int), scores)

# Alert fatigue: false alerts per analyst-hour if this test set were one day of traffic
false_alerts = int(((scores >= 0.5) & (y_test.values == 0)).sum())
analyst_hours = 8 * 6  # six analysts, one shift
false_alerts_per_analyst_hour = false_alerts / analyst_hours

# Robustness (Labs 12, 17)
columns = available_numeric_attack_features(X_train)
bounds = training_bounds(X_train, columns)
stressed = bounded_random_perturbation(X_test, columns=columns, epsilon=0.10, bounds=bounds, random_state=0)
stressed_recall = binary_metrics(y_test, detector.predict(stressed)).recall

# LLM application (Lab 23)
injection_results, _ = evaluate_controls(build_test_cases(), ControlConfig(True, True, True, True, True))
attack_success = injection_results.loc[injection_results["kind"] == "attack", "attack_succeeded"].mean()
benign_success = injection_results.loc[injection_results["kind"] == "benign", "benign_completed"].mean()

# Privacy (Lab 26)
_, logs = simulate_assistant_logging(build_test_cases(), ControlConfig(output_secret_filter=True), log_system_prompt=False, redact=True)
canary_hits = int(scan_for_canaries([str(r) for r in logs], [CANARY_SECRET])["hits"].iloc[0])

# Red-team regression (Lab 27): the version currently deployed
deployed_version = list(APP_VERSIONS)[1]
campaign = run_campaign(default_cases(), deployed_version)
regression_pass = campaign["passed"].mean()
unnoticed_failures = int((~campaign["passed"] & (campaign["monitoring_noticed"] != True)).sum())  # noqa: E712

measurements = pd.Series({
    "recall": detection.recall,
    "precision": detection.precision,
    "false_alerts_per_analyst_hour": false_alerts_per_analyst_hour,
    "recall_under_stress": stressed_recall,
    "injection_attack_success": attack_success,
    "benign_task_success": benign_success,
    "canary_hits_in_logs": canary_hits,
    "red_team_pass_rate": regression_pass,
    "unnoticed_red_team_failures": unnoticed_failures,
})
display(measurements.round(3).to_frame("value"))

## Step 3: Attach a baseline, threshold, owner, and decision

Baselines here stand for "last quarter". Thresholds are the line where someone must act.

In [ ]:
metrics = [
    Metric("recall", "detection", "share of real attacks alerted", measurements["recall"],
           baseline=0.97, threshold=0.95, better="higher", owner="detection engineering",
           decision_if_breached="investigate which categories regressed (Lab 11) before any retrain"),
    Metric("false alerts per analyst-hour", "detection", "benign flows alerted / analyst hours", measurements["false_alerts_per_analyst_hour"],
           baseline=60, threshold=80, better="lower", owner="SOC manager",
           decision_if_breached="re-select threshold on validation data (Lab 09); do not lower recall target silently"),
    Metric("recall under bounded stress", "robustness", "attack recall with epsilon=0.10 input noise", measurements["recall_under_stress"],
           baseline=0.95, threshold=0.90, better="higher", owner="ML engineering",
           decision_if_breached="compare mitigations (Lab 17) and retest"),
    Metric("prompt-injection attack success", "LLM application", "share of red-team payloads that cause a harmful outcome", measurements["injection_attack_success"],
           baseline=0.0, threshold=0.0, better="lower", owner="AI platform security",
           decision_if_breached="block the release; add the payload to the regression suite"),
    Metric("legitimate task success", "LLM application", "benign requests completed under all controls", measurements["benign_task_success"],
           baseline=1.0, threshold=0.95, better="higher", owner="product owner",
           decision_if_breached="review which control over-blocks; safety that breaks the job gets bypassed"),
    Metric("canary hits in logs", "privacy", "planted secrets found in retained logs", measurements["canary_hits_in_logs"],
           baseline=0, threshold=0, better="lower", owner="privacy engineering",
           decision_if_breached="purge affected logs, rotate the secret, fix the logging path"),
    Metric("red-team regression pass rate", "operations", "share of red-team cases passing on the deployed version", measurements["red_team_pass_rate"],
           baseline=0.92, threshold=1.0, better="higher", owner="AI red team",
           decision_if_breached="ship the pending fix (Lab 27 v3) and rerun the full suite"),
    Metric("unnoticed red-team failures", "operations", "failing cases that monitoring did not record", measurements["unnoticed_red_team_failures"],
           baseline=1, threshold=0, better="lower", owner="security monitoring",
           decision_if_breached="add a log event for that failure path before closing the finding"),
]

card = scorecard(metrics)
display(card[["status", "layer", "name", "value", "baseline", "threshold", "owner", "decision_if_breached"]].round(3))

## Step 4: Read the scorecard as next actions

In [ ]:
for row in card[card["status"] == "BREACHED"].itertuples():
    print(f"- {row.name} = {row.value:.3g} (threshold {row.threshold}) -> {row.owner}: {row.decision_if_breached}")

## Interpretation

1. Which breached metric is the most urgent, and who acts on it?
2. Why does "false alerts per analyst-hour" belong next to recall instead of on a separate dashboard?
3. "Prompts blocked" is a motion metric. Which outcome metric replaces it here?
4. A metric with no owner and no decision attached: what happens to it in practice?

**Takeaway:** measure outcomes, not motion. A useful metric has a baseline, a threshold, an owner, and a decision attached to it.